## US Core Examples Transaction Bundle Maker
### bundle all resources in folder as a transaction using PUT interactions

- python version 3.10.2
- Use Python Dict
- get data from folder
- add meta tag "Argo-R4-R6"
- bundle 
  - limit size to 100 entries
  - make >1 bundle if larger
- update and resolve ids
- validate and save

In [1]:
# from fhir.resources import construct_fhir_element
from json import load, dumps, loads
from IPython import display as D
from requests import get, post, put
from IPython.display import display, Markdown, HTML
# #from fhirclient.r4models import bundle as B
# #import fhirclient.r4models.fhirdate as D
# import os, uuid
from datetime import datetime, timezone
from pathlib import Path

bundle_id = "Argo-R4-R6-Examples"
bundle_type = 'transaction'
base_url = 'https://anjjwdewgb.edge.aidbox.app/fhir'   # no trailing '/'
new_id_prefix = ''
my_tag = "Argo-R4-R6"
in_path = Path(r'/Users/ehaas/.fhir/packages/hl7.fhir.us.core#dev/package/example') #'R6!'
out_path =Path(r'argo-r4-r6')


###  write to file

In [2]:
def write_file(r): # write file
    out_file = out_path / f"Bundle-{bundle_id}.json"
    print(f'writing to ... {out_file}') 
    out_file.write_text(r)


### create Bundle 'b'  change the id for unique Bundles!!!

In [3]:
dt = datetime.utcnow()
timestamp = dt.replace(tzinfo = timezone.utc)
def create_bundle(i):
    
  my_bundle =dict(
  resourceType = 'Bundle',
  id =  f'{bundle_id}-{i//99+1}',
  type = bundle_type,
  timestamp = timestamp.isoformat(),
  )

  b = dumps(my_bundle, indent=2)
  print(f'my_bundle = {b}')
  return(my_bundle)


### Resort and save as JSON

In [4]:
def save_bundle(my_bundle):
  transaction_order = dict(
              Patient = 0,
              Organization = 1,
              Practitioner = 2,
              Location = 4,
              PractitionerRole = 5,
              Specimen = 6,
              Device = 7,
              Encounter = 8,
              DocumentReference = 9,
              Observation = 10,
              Media = 11,
              Medication = 12,
              RelatedPerson = 13,
              DeviceAssociation = 14
              )
  my_bundle['entry'].sort(key=lambda x: transaction_order.get(x['resource']['resourceType'],14))
  out_file = out_path / f'{my_bundle["id"]}.json'
  print(f'writing bundle with {len(my_bundle["entry"])} entries to {out_file} ....')
  out_file.write_text(dumps(my_bundle, indent=2))

### Add resources to bundle

 - add tag to each resource first
 - remove text elements
  
### Resort bundle for transaction
 - resort for transaction to make sure all the referred to resources are before the referring ones


In [5]:
# from fhir.resources.fhirtypesvalidators import ValidationError
# my_bundle['entry'] = []
# type_list = []
# failed_list = []
resource_table = []


def get_entry(r):
  ref = f'{r["resourceType"]}/{r["id"]}'
  r.pop('text', None) # remove text elements from Bundle
  try:  # add meta tag
        r['meta']['tag'].append({"code":my_tag,})
  except KeyError:  #no tag element
        r['meta']['tag']= [{"code":my_tag,}] # add meta tag

#======== remove references to location and servicerequest and questionnaireresponse ========
  if r["resourceType"] == 'Encounter':
    # print('=============Encounter:  remove location reference ============')
    # print(f"==============Encounter.location = {r['location'][0]['location']}=============")
    try:
      my_display = r['location'][0]['location'].pop('reference')
    except KeyError as e:
      pass
    else:
      try:
         r['location'][0]['location']['display']
      except KeyError as e:
         r['location'][0]['location']['display']= my_display
    # print('=============Encounter location reference removed !! ============')
    # print(f"==============Encounter.location = {r['location'][0]}=============")
  if r["resourceType"] == 'Procedure': 
    try:
      my_display = r['basedOn'][0].pop('reference')
    except KeyError as e:
      pass
    else:
      try:
         r['basedOn'][0]['display']
      except KeyError as e:
         r['basedOn'][0]['display']= my_display

  if r["resourceType"] == 'Observation': 
    # print('=============Encounter:  remove derivedFrom reference ============')
    # print(f"==============Encounter.derivedFrom = {r['derivedFrom'][0]}=============")
    try:
      my_display = r['derivedFrom'][0].pop('reference')
    except KeyError as e:
      pass
    else:
      try:
         r['derivedFrom'][0]['display']
      except KeyError as e:
         r['derivedFrom'][0]['display']= my_display
    # print('=============Encounter derivedFrom reference removed !! ============')
    # print(f"==============Encounter.derivedFrom = {r['derivedFrom'][0]}=============")

    # ========= end custom reference removal ===========
  e = dict(
      resource = r,
      fullUrl = f'{base_url}/{ref}'
      )
  if bundle_type in ['transaction', 'batch']:
      e['request'] = dict(
                  method = 'PUT',
                  url = ref,
                  )
  resource_table.append((e['fullUrl'],r["resourceType"]))
  return(e)

for i, f in enumerate(in_path.glob("*.json")):
    if i % 99 == 0:
      try:
         save_bundle(my_bundle)
      except NameError as e:
         pass
      my_bundle = create_bundle(i)
      my_bundle['entry'] = []
    # print(f"========== {i}, {f.stem}=========")
    my_obj = loads(f.read_text())
    try:
      my_resourceType =  my_obj["resourceType"]
    except KeyError as e:
      print(f'Keyerror : {e} for {i}, {f.stem}.json file')
    else:
      if my_resourceType in ["Bundle"]: # decompose bundle
          for my_entry in my_obj['entry']:
              my_resource = my_entry['resource']
              my_bundle['entry'].append(get_entry(my_resource))
              
      else:
          my_bundle['entry'].append(get_entry(my_obj))
save_bundle(my_bundle)

# print(dumps(my_bundle, indent=2))

my_bundle = {
  "resourceType": "Bundle",
  "id": "Argo-R4-R6-Examples-1",
  "type": "transaction",
  "timestamp": "2025-08-19T21:25:47.082075+00:00"
}
Keyerror : 'resourceType' for 6, .index.json file
writing bundle with 100 entries to argo-r4-r6/Argo-R4-R6-Examples-1.json ....
my_bundle = {
  "resourceType": "Bundle",
  "id": "Argo-R4-R6-Examples-2",
  "type": "transaction",
  "timestamp": "2025-08-19T21:25:47.082075+00:00"
}
writing bundle with 72 entries to argo-r4-r6/Argo-R4-R6-Examples-2.json ....


### Validate using Java CLI

see profile validator doco here: https://confluence.hl7.org/spaces/FHIR/pages/35718580/Using+the+FHIR+Validator

same as IG output so not used for now...

### print out FHIR_id Mapping table

In [6]:
my_table = '|Type|FHIR ID|\n|---|---|'
for i in sorted(resource_table):
    my_table = (f"{my_table}\n| {i[0]} | {i[1]} |")

display(Markdown(my_table))
print(my_table)


|Type|FHIR ID|
|---|---|
| https://anjjwdewgb.edge.aidbox.app/fhir/AllergyIntolerance/example | AllergyIntolerance |
| https://anjjwdewgb.edge.aidbox.app/fhir/AllergyIntolerance/non-pharmacologic-agent-example | AllergyIntolerance |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/condition-SDOH-example | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/condition-duodenal-ulcer | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/encounter-diagnosis-example1 | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/encounter-diagnosis-example2 | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/health-concern-example | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Device/udi-1 | Device |
| https://anjjwdewgb.edge.aidbox.app/fhir/Device/udi-2 | Device |
| https://anjjwdewgb.edge.aidbox.app/fhir/Device/udi-3 | Device |
| https://anjjwdewgb.edge.aidbox.app/fhir/DeviceAssociation/udi-1 | DeviceAssociation |
| https://anjjwdewgb.edge.aidbox.app/fhir/DeviceAssociation/udi-2 | DeviceAssociation |
| https://anjjwdewgb.edge.aidbox.app/fhir/DeviceAssociation/udi-3 | DeviceAssociation |
| https://anjjwdewgb.edge.aidbox.app/fhir/DocumentReference/adi-dnr | DocumentReference |
| https://anjjwdewgb.edge.aidbox.app/fhir/DocumentReference/discharge-summary | DocumentReference |
| https://anjjwdewgb.edge.aidbox.app/fhir/DocumentReference/episode-summary | DocumentReference |
| https://anjjwdewgb.edge.aidbox.app/fhir/DocumentReference/living-will | DocumentReference |
| https://anjjwdewgb.edge.aidbox.app/fhir/DocumentReference/polst | DocumentReference |
| https://anjjwdewgb.edge.aidbox.app/fhir/Encounter/1036 | Encounter |
| https://anjjwdewgb.edge.aidbox.app/fhir/Encounter/delivery | Encounter |
| https://anjjwdewgb.edge.aidbox.app/fhir/Encounter/example-1 | Encounter |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/10-minute-apgar-color | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/10-minute-apgar-heart-rate | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/10-minute-apgar-muscle-tone | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/10-minute-apgar-reflex-irritability | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/10-minute-apgar-respiratory-effort | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/10-minute-apgar-score | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/10-minute-apgar-score-panel | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/ADI-example | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/AHC-HRSN-item-example-68517-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/AUDIT-C-item-example-68517-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/AUDIT-C-item-example-68519-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/AUDIT-C-item-example-68520-6 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/AUDIT-C-item-example-75626-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/AUDIT-C-panel-example-72109-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/EVS-item-example-68516-4 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/EVS-item-example-89555-7 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/EVS-panel-example-89574-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/HVS-item-example-88122-7 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/HVS-item-example-88123-5 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/HVS-item-example-88124-3 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/HVS-panel-example-88121-9 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44250-9 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44251-7 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44252-5 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44253-3 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44254-1 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44255-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44258-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44259-0 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44260-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-44261-6 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-item-example-69722-7 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PHQ9-panel-example-44249-1 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-54899-0 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-56051-6 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-56799-0 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-63512-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-63586-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-67875-5 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-71802-3 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-76437-3 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-76501-6 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-82589-3 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93026-3 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93027-1 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93028-9 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93029-7 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93030-5 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93033-9 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93034-7 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93035-4 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-item-example-93038-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-multiselect-item-example-32624-9-answer0 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-multiselect-item-example-32624-9-answer1 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-multiselect-item-example-93031-3-answer0 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-multiselect-item-example-93031-3-answer1 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-panel-example-93025-5 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-panel-example-93039-6 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-panel-example-93040-4 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-panel-example-93041-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-panel-example-93042-0 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/PRAPARE-panel-example-93043-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/TAPS-item-example-75889-6 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/TAPS-item-example-88037-7 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/TAPS-item-example-96842-0 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/TAPS-item-example-96843-8 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/TAPS-item-example-96844-6 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/TAPS-panel-example-96841-2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/alcohol-use-status | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/alcoholic-drinks-per-day | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/at-home-in-vitro-test | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/average-blood-pressure | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/blood-pressure | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/bmi | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/bp-data-absent | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/capillary-refill-time-nail-bed | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/care-experience-preference | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-erythrocytes | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-hematocrit | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-hemoglobin | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-leukocytes | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-mch | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-mchc | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-mcv | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/cbc-platelets | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/dxa-femur-l-armass-bmd | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/dxa-femur-l-t-score-bmd | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/dxa-femur-l-z-score-bmd | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/dxa-hip-l-armass-bmd | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/dxa-hip-l-t-score-bmd | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/dxa-hip-l-z-score-bmd | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/ekg-impression | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/ekg-lead | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/exercise-per-day | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/exercise-per-week | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/head-circumference | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/heart-rate | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/heart-rate-rhythm | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/height | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/jugular-vein-distension | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/length | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/no-ADI-example | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/observation-occupation | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/observation-occupation-industry-unknown | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/observation-occupation-unknown | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/ofc-percentile | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/oxygen-saturation | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/p-r-interval-ekg-lead | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/pack-years-example | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/pediatric-bmi-example | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/pediatric-wt-example | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/pregnancy-intent | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/pregnancy-status | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/pulse-intensity-palpation | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/q-t-interval-ekg-lead | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/qrs-dur-ekg-lead | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/respiratory-rate | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/satO2-fiO2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-bun | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-calcium | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-chloride | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-co2 | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-creatinine | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-glucose | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-potassium | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/serum-sodium | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/simple-observation-cognitive-status | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/simple-observation-disability-status | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/simple-observation-functional-status | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/simple-observation-sdoh | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/some-day-smoker | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/substance-use-status | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/temperature | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/treatment-intervention-preference | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/urobilinogen | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/weight | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/xray-chest-findings | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Observation/xray-chest-impression | Observation |
| https://anjjwdewgb.edge.aidbox.app/fhir/Patient/child-example | Patient |
| https://anjjwdewgb.edge.aidbox.app/fhir/Patient/deceased-example | Patient |
| https://anjjwdewgb.edge.aidbox.app/fhir/Patient/example | Patient |
| https://anjjwdewgb.edge.aidbox.app/fhir/Patient/infant-example | Patient |
| https://anjjwdewgb.edge.aidbox.app/fhir/Practitioner/practitioner-1 | Practitioner |
| https://anjjwdewgb.edge.aidbox.app/fhir/Practitioner/practitioner-2 | Practitioner |
| https://anjjwdewgb.edge.aidbox.app/fhir/Practitioner/practitioner-pharmacist | Practitioner |
| https://anjjwdewgb.edge.aidbox.app/fhir/Procedure/defib-implant | Procedure |
| https://anjjwdewgb.edge.aidbox.app/fhir/Procedure/rehab | Procedure |
| https://anjjwdewgb.edge.aidbox.app/fhir/Specimen/example-serum-lipemic | Specimen |
| https://anjjwdewgb.edge.aidbox.app/fhir/Specimen/specimen-example-serum | Specimen |
| https://anjjwdewgb.edge.aidbox.app/fhir/Specimen/specimen-example-urine | Specimen |
| https://anjjwdewgb.edge.aidbox.app/fhir/Specimen/specimen-example-whole-blood | Specimen |

|Type|FHIR ID|
|---|---|
| https://anjjwdewgb.edge.aidbox.app/fhir/AllergyIntolerance/example | AllergyIntolerance |
| https://anjjwdewgb.edge.aidbox.app/fhir/AllergyIntolerance/non-pharmacologic-agent-example | AllergyIntolerance |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/condition-SDOH-example | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/condition-duodenal-ulcer | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/encounter-diagnosis-example1 | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/encounter-diagnosis-example2 | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Condition/health-concern-example | Condition |
| https://anjjwdewgb.edge.aidbox.app/fhir/Device/udi-1 | Device |
| https://anjjwdewgb.edge.aidbox.app/fhir/Device/udi-2 | Device |
| https://anjjwdewgb.edge.aidbox.app/fhir/Device/udi-3 | Device |
| https://anjjwdewgb.edge.aidbox.app/fhir/DeviceAssociation/udi-1 | DeviceAssociation |
| https://anjjwdewgb.e